In [1]:
!pip install sentence-transformers torch python-docx openpyxl pandas wandb dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.4 MB/s eta 0:00:00


In [3]:
import json
import os
import wandb
from datetime import datetime
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)

# 1. Khởi tạo Wandb
wandb.init(
    project="law-it-fine-tune-v2",
    name=f"huyydangg-dek21-80-20-10epochs-{datetime.now().strftime('%H%M')}",
    config={
        "model": "huyydangg/DEk21_hcmute_embedding",
        "epochs": 10,
        "batch_size": 16,
        "save_steps": 100,
        "train_split": 0.8
    }
)

def load_and_split_data(jsonl_path):
    """
    Đọc dữ liệu từ law_chunks_hier.jsonl và chia 80/20.
    """
    anchors = []
    positives = []

    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(f"Không tìm thấy file tại: {jsonl_path}")

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                p = data.get("payload", {})

                ten_van_ban = p.get('ten_van_ban', '').strip()
                dieu_ten = p.get('dieu_ten', '').strip()
                dieu_so = p.get('dieu_so', '').strip()

                # Positive là nội dung kiến thức chi tiết
                positive = p.get("noi_dung_chunk", "").strip()

                # Cấu trúc Anchor dựa trên phân cấp văn bản
                if ten_van_ban and dieu_ten and positive:
                    anchor = f"Quy định về {dieu_ten} (Điều {dieu_so}) trong {ten_van_ban}".strip()
                    anchors.append(anchor)
                    positives.append(positive)
            except json.JSONDecodeError:
                continue

    # Tạo Dataset từ dictionary
    full_dataset = Dataset.from_dict({"anchor": anchors, "positive": positives})

    # Chia 80% train và 20% test (validation)
    split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)
    return split_dataset["train"], split_dataset["test"]

# Xác định đường dẫn file
data_path = 'data/law_chunks_hier.jsonl' if os.path.exists('data/law_chunks_hier.jsonl') else 'law_chunks_hier.jsonl'

train_set, test_set = load_and_split_data(data_path)

# 2. Khởi tạo model đích
model = SentenceTransformer('huyydangg/DEk21_hcmute_embedding')

# 3. Sử dụng MultipleNegativesRankingLoss
train_loss = losses.MultipleNegativesRankingLoss(model)

output_dir = f"./output/law_v2_{datetime.now().strftime('%Y%m%d_%H%M')}"

# 4. Thiết lập tham số huấn luyện
args = SentenceTransformerTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=20,
    report_to="wandb",
    save_total_limit=2,
    load_best_model_at_end=True,
)

# 5. Khởi tạo Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_set,
    eval_dataset=test_set,
    loss=train_loss,
)

print("🚀 Bắt đầu quá trình huấn luyện...")
trainer.train()

# 6. Lưu mô hình và đồng bộ Wandb
model.save_pretrained(f"{output_dir}/final_model_v2")
wandb.finish()

print(f"✅ Hoàn tất! Tập huấn luyện: {len(train_set)}, Tập kiểm thử: {len(test_set)}")
print(f"✅ Mô hình tốt nhất đã được lưu tại: {output_dir}/final_model_v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

eval/loss,█▆▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,▆▆▆▃▂▇▁▁▃▄▂▇▂▇▁▁▄▅▇▁▅▇▅▃▁█▁█▂▁▇▆▇▁▂▃▁█▆█
eval/samples_per_second,▃▃▂▅▇▂██▆▅▇▂▇▂██▅▃▂█▃▂▄▆█▁█▁▇█▁▃▂█▇▆█▁▃▁
eval/steps_per_second,▃▃▂▅▇▂██▆▅▇▂▇▂██▅▃▂█▃▂▄▆█▁█▁▇█▁▃▂█▇▆█▁▃▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇█████
train/grad_norm,▆█▅▅▅▂▃▄▁▃▃▃▅▄▄▂▄▃▁▂▄▂▂▂▁▃▄▄▁▃▁▁▁▂▁▂▂▂▄▂
train/learning_rate,▂▂▆█████▇▇▆▆▆▆▆▆▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁
train/loss,█▆▅▃▂▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.23793
eval/runtime,6.6493


✅ Hoàn tất! Tập huấn luyện: 7526, Tập kiểm thử: 1882
✅ Mô hình tốt nhất đã được lưu tại: ./output/law_v2_20260505_1339/final_model_v2


In [4]:
import shutil
import os
from google.colab import files
from datetime import datetime

# Define paths
final_model_path = os.path.join(output_dir, 'final_model_v2')
zip_filename = f"law_v2_model_{datetime.now().strftime('%Y%m%d_%H%M')}"

if os.path.exists(final_model_path):
    print(f"📦 Zipping model folder: {final_model_path}...")
    # Create zip file
    shutil.make_archive(zip_filename, 'zip', final_model_path)

    full_zip_path = f"{zip_filename}.zip"
    print(f"✅ Zip created: {full_zip_path}")

    # Download the file
    print("💾 Starting download...")
    files.download(full_zip_path)
else:
    print(f"❌ Error: The directory {final_model_path} was not found. Please ensure training is complete.")

📦 Zipping model folder: ./output/law_v2_20260505_1339/final_model_v2...
✅ Zip created: law_v2_model_20260505_1418.zip
💾 Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>